In [19]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    !pip install -q kaggle lightgbm

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'


walmart-recruiting-store-sales-forecasting.zip: Skipping, found more recently modified local copy (use --force to force download)


In [20]:
import numpy as np
import pandas as pd

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)


In [21]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv', parse_dates=['Date'])
sample = pd.read_csv(f'{DATA_DIR}/sampleSubmission.csv')

y_train = train['Weekly_Sales'].clip(lower=0)
df = train.assign(Weekly_Sales=y_train)
train.shape, test.shape, sample.shape


((421570, 5), (115064, 4), (115064, 2))

In [22]:
# CHECK 1: does our submission Id format exactly match sampleSubmission?
our_ids = (test['Store'].astype(str) + '_' + test['Dept'].astype(str) + '_'
           + test['Date'].dt.strftime('%Y-%m-%d'))
sample_ids = sample['Id']

print('our id count   :', len(our_ids))
print('sample id count:', len(sample_ids))
print('exact set match:', set(our_ids) == set(sample_ids))
print('same row order :', bool((our_ids.values == sample_ids.values).all()))
print('in sample not ours:', len(set(sample_ids) - set(our_ids)),
      '| in ours not sample:', len(set(our_ids) - set(sample_ids)))
print('example sample Id:', sample_ids.iloc[0], '| example our Id:', our_ids.iloc[0])


our id count   : 115064
sample id count: 115064
exact set match: True
same row order : True
in sample not ours: 0 | in ours not sample: 0
example sample Id: 1_1_2012-11-02 | example our Id: 1_1_2012-11-02


In [23]:
# naive baseline: predict the same (store, dept) sales from exactly 52 weeks earlier
def naive_lag52(history, target):
    hist = history.set_index(['Store', 'Dept', 'Date'])['Weekly_Sales']
    series_mean = history.groupby(['Store', 'Dept'])['Weekly_Sales'].mean()
    lag_idx = pd.MultiIndex.from_arrays(
        [target['Store'], target['Dept'], target['Date'] - pd.Timedelta(weeks=52)])
    preds = pd.Series(hist.reindex(lag_idx).values, index=target.index)
    fallback = pd.Series(target.set_index(['Store', 'Dept']).index.map(series_mean), index=target.index)
    return preds.fillna(fallback).fillna(history['Weekly_Sales'].mean()).values


In [24]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])

# CHECK 2: naive baseline on each original CV fold. note which folds contain a Nov/Dec peak
print('naive lag-52 WMAE on the original CV folds:')
for i, (te, ve) in enumerate(splits):
    tr = df[df['Date'] <= te]
    va = df[(df['Date'] > te) & (df['Date'] <= ve)]
    p = naive_lag52(tr, va)
    has_peak = bool(va['Date'].dt.month.isin([11, 12]).any())
    score = wmae(va['Weekly_Sales'].values, p, va['IsHoliday'].values)
    print(f'  fold {i}: val ends {pd.Timestamp(ve).date()}  WMAE={score:8.1f}  has Nov/Dec: {has_peak}')


naive lag-52 WMAE on the original CV folds:
  fold 0: val ends 2011-06-03  WMAE=  3643.6  has Nov/Dec: True
  fold 1: val ends 2012-02-03  WMAE=  2037.5  has Nov/Dec: True
  fold 2: val ends 2012-10-05  WMAE=  1832.4  has Nov/Dec: False


In [25]:
# CHECK 3: a holdout that mimics the Kaggle test calendar (train to late Oct, predict next Nov->Jul)
CUTOFF = pd.Timestamp('2011-10-28')
VAL_END = pd.Timestamp('2012-07-27')
tr = df[df['Date'] <= CUTOFF]
va = df[(df['Date'] > CUTOFF) & (df['Date'] <= VAL_END)]
p = naive_lag52(tr, va)
score = wmae(va['Weekly_Sales'].values, p, va['IsHoliday'].values)

print('test-mimicking holdout:')
print(f'  val window: {va["Date"].min().date()} .. {va["Date"].max().date()}  ({va["Date"].nunique()} weeks, n={len(va)})')
print(f'  includes Nov/Dec peak: {bool(va["Date"].dt.month.isin([11, 12]).any())}')
print(f'  naive lag-52 WMAE = {score:.1f}')
print()
print('reference: Kaggle test window is 2012-11-02 .. 2013-07-26, your naive Kaggle score ~2800')
print('if this holdout WMAE is ~2800 too, the holdout faithfully mimics the real test')


test-mimicking holdout:
  val window: 2011-11-04 .. 2012-07-27  (39 weeks, n=115856)
  includes Nov/Dec peak: True
  naive lag-52 WMAE = 2074.3

reference: Kaggle test window is 2012-11-02 .. 2013-07-26, your naive Kaggle score ~2800
if this holdout WMAE is ~2800 too, the holdout faithfully mimics the real test


In [26]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])
FEATURE_COLS = ['Store', 'Dept', 'IsHoliday', 'Size', 'Type', 'Temperature', 'Fuel_Price', 'CPI',
                'Unemployment', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
                'Year', 'Month', 'Week', 'IsPreChristmas']

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        d = X.copy()
        d['Date'] = pd.to_datetime(d['Date'])
        d = d.merge(self.stores_df, on='Store', how='left')
        d = d.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')
        d[MARKDOWN_COLS] = d[MARKDOWN_COLS].fillna(0)
        d['Type'] = d['Type'].map({'A': 0, 'B': 1, 'C': 2})
        d['IsHoliday'] = d['IsHoliday'].astype(int)
        d['Year'] = d['Date'].dt.year
        d['Month'] = d['Date'].dt.month
        d['Week'] = d['Date'].dt.isocalendar().week.astype(int)
        d['IsPreChristmas'] = d['Date'].isin(PRE_CHRISTMAS).astype(int)
        return d[FEATURE_COLS]

def score_lgbm(tr, va):
    pipe = Pipeline([
        ('features', FeatureBuilder(stores, features)),
        ('model', LGBMRegressor(n_estimators=1000, num_leaves=127, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, verbose=-1)),
    ])
    pipe.fit(tr[['Store', 'Dept', 'Date', 'IsHoliday']], tr['Weekly_Sales'])
    pr = pipe.predict(va[['Store', 'Dept', 'Date', 'IsHoliday']])
    return wmae(va['Weekly_Sales'].values, pr, va['IsHoliday'].values)

# CHECK 4: a real model on the easy fold 2 vs the test-mimicking holdout
te2, ve2 = splits[2]
f2_tr = df[df['Date'] <= te2]
f2_va = df[(df['Date'] > te2) & (df['Date'] <= ve2)]

print('LightGBM (v1 params):')
print(f'  fold 2 (no Nov/Dec peak)      WMAE = {score_lgbm(f2_tr, f2_va):.1f}')
print(f'  test-mimic holdout (Nov->Jul) WMAE = {score_lgbm(tr, va):.1f}')


LightGBM (v1 params):
  fold 2 (no Nov/Dec peak)      WMAE = 2265.8
  test-mimic holdout (Nov->Jul) WMAE = 2789.3
